#### Dataset Selection and Industry Context
##
#### **Industry**: E-Commerce
#
#### **Dataset**: Brazilian E-Commerce Public Dataset by Olist
####   **Source**  : Kaggle kaggle.com/datasets/olistbr/brazilian-ecommerce
####   **Size**   : ~100,000 real orders from 2016 to 2018
####   **Files**   : 7 CSV files (orders, items, customers, payments,reviews, products, category translations)
#
#### Business Problem:
- #####   Can we predict whether a customer will leave a negative review based on order features like delivery time, price, and category?
#
#### Why This Matters:
- #####   Negative reviews damage seller reputation and reduce repeat sales.
- #####   Early prediction lets sellers take corrective action.
#
#### How Analytics Helps:
- #####   Using Spark ML on delivery time, price, delivery, and product
- #####   category data, we build a classifier that identifies high-risk
- #####   orders before a review is submitted.

### Data Loading

In [0]:
# LOAD ALL 7 CSV FILES

orders      = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_orders_dataset.csv',
                             header=True, inferSchema=True)
items       = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_order_items_dataset.csv',
                             header=True, inferSchema=True)
customers   = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_customers_dataset.csv',
                             header=True, inferSchema=True)
payments    = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_order_payments_dataset.csv',
                             header=True, inferSchema=True)
reviews     = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_order_reviews_dataset.csv',
                             header=True, inferSchema=True)
products    = spark.read.csv('/Volumes/workspace/default/dataset_olist/olist_products_dataset.csv',
                             header=True, inferSchema=True)
translation = spark.read.csv('/Volumes/workspace/default/dataset_olist/product_category_name_translation.csv',
                             header=True, inferSchema=True)

print(f'Orders:      {orders.count():,} rows')
print(f'Items:       {items.count():,} rows')
print(f'Customers:   {customers.count():,} rows')
print(f'Payments:    {payments.count():,} rows')
print(f'Reviews:     {reviews.count():,} rows')
print(f'Products:    {products.count():,} rows')

In [0]:
# EXPLORE THE DATA

print('=== ORDERS SCHEMA ===')
orders.printSchema()

print('=== ORDERS FIRST 5 ROWS ===')
orders.show(5)

# Count null values in orders
from pyspark.sql.functions import col, sum as spark_sum
null_counts = orders.select([
  spark_sum(col(c).isNull().cast('int')).alias(c)
  for c in orders.columns
])
print('=== NULL VALUE COUNT PER COLUMN ===')
null_counts.show()

### Spark Processing

In [0]:
# DATA CLEANING

from pyspark.sql.functions import to_timestamp, datediff, col

# ORDERS: keep only delivered orders
orders_clean = orders.filter(col('order_status') == 'delivered')
print(f'Delivered orders: {orders_clean.count():,}')

# Drop rows with missing timestamps
orders_clean = orders_clean.dropna(subset=[
    'order_purchase_timestamp',
    'order_delivered_customer_date'
])
print(f'After dropping nulls: {orders_clean.count():,}')

# Remove duplicate order IDs
orders_clean = orders_clean.dropDuplicates(['order_id'])

# Calculate delivery days
orders_clean = orders_clean.withColumn(
    'delivery_days',
    datediff(
        to_timestamp(col('order_delivered_customer_date'), 'M/d/yyyy H:mm'),
        to_timestamp(col('order_purchase_timestamp'), 'M/d/yyyy H:mm')
    )
)

# Remove invalid delivery times
orders_clean = orders_clean.filter(col('delivery_days') > 0)
print(f'Final clean orders: {orders_clean.count():,}')

# REVIEWS: drop null review scores, one review per order
reviews_clean = reviews.dropna(subset=['review_score', 'order_id'])
reviews_clean = reviews_clean.dropDuplicates(['order_id'])
print(f'Reviews after cleaning: {reviews_clean.count():,}')

# PAYMENTS: drop null or zero payment values
payments_clean = payments.dropna(subset=['payment_value', 'order_id'])
payments_clean = payments_clean.filter(col('payment_value') > 0)
print(f'Payments after cleaning: {payments_clean.count():,}')

# ITEMS: drop null prices or negative freight values
items_clean = items.dropna(subset=['price', 'freight_value', 'order_id'])
items_clean = items_clean.filter((col('price') > 0) & (col('freight_value') >= 0))
print(f'Items after cleaning: {items_clean.count():,}')

# PRODUCTS: fill null category with 'unknown'
products_clean = products.fillna({'product_category_name': 'unknown'})
print(f'Products after cleaning: {products_clean.count():,}')

# CUSTOMERS: drop rows with null customer_state
customers_clean = customers.dropna(subset=['customer_state'])
print(f'Customers after cleaning: {customers_clean.count():,}')

In [0]:
# JOIN ALL TABLES INTO MASTER DATAFRAME

from pyspark.sql.functions import avg, sum as spark_sum, count, desc
from pyspark.sql.functions import round as spark_round

# Join orders + reviews (using reviews_clean)
df = orders_clean.join(
    reviews_clean.select('order_id', 'review_score'),
    on='order_id', how='inner'
)

# Aggregate payments per order (using payments_clean)
pay_agg = payments_clean.groupBy('order_id').agg(
    avg('payment_value').alias('avg_payment'),
    avg('payment_installments').alias('payment_installments')
)
df = df.join(pay_agg, on='order_id', how='left')

# Aggregate items per order (using items_clean)
items_agg = items_clean.groupBy('order_id').agg(
    spark_sum('price').alias('total_price'),
    spark_sum('freight_value').alias('freight_value'),
    count('order_item_id').alias('item_count')
)
df = df.join(items_agg, on='order_id', how='left')

# Add customer state (using customers_clean)
df = df.join(
    customers_clean.select('customer_id', 'customer_state'),
    on='customer_id', how='left'
)

# Add product category in English (using items_clean and products_clean)
items_cat = items_clean.join(
    products_clean.select('product_id', 'product_category_name'),
    on='product_id', how='left'
).join(translation, on='product_category_name', how='left'
).select('order_id', 'product_category_name_english'
).dropDuplicates(['order_id'])

df = df.join(items_cat, on='order_id', how='left')

# Drop any remaining nulls in key feature columns used for ML later
df = df.dropna(subset=['total_price', 'freight_value',
                        'payment_installments', 'item_count',
                        'delivery_days', 'review_score'])

print(f'Master DataFrame: {df.count():,} rows, {len(df.columns)} columns')
df.show(3)

### Analytics

In [0]:
# ANALYTICS: REVENUE BY PRODUCT CATEGORY

revenue_by_cat = df.groupBy('product_category_name_english').agg(
  spark_round(spark_sum('total_price'), 2).alias('total_revenue'),
  count('order_id').alias('total_orders'),
  spark_round(avg('total_price'), 2).alias('avg_order_value')
).orderBy(desc('total_revenue'))

print('=== TOP 10 PRODUCT CATEGORIES BY REVENUE ===')
revenue_by_cat.show(10)

In [0]:
# ANALYTICS: MONTHLY ORDER TRENDS

from pyspark.sql.functions import date_format, to_timestamp

monthly = df.withColumn(
    'month',
    date_format(
        to_timestamp(col('order_purchase_timestamp'), 'M/d/yyyy H:mm'),  # format added
        'yyyy-MM'
    )
).groupBy('month').agg(
    count('order_id').alias('order_count'),
    spark_round(spark_sum('total_price'), 2).alias('monthly_revenue')
).orderBy('month')

print('=== MONTHLY ORDER VOLUME AND REVENUE ===')
monthly.show(24)

In [0]:
# ANALYTICS: AVERAGE DELIVERY TIME BY STATE

delivery_by_state = df.groupBy('customer_state').agg(
  spark_round(avg('delivery_days'), 1).alias('avg_delivery_days'),
  count('order_id').alias('total_orders')
).orderBy(desc('avg_delivery_days'))

print('=== DELIVERY PERFORMANCE BY STATE (Slowest first) ===')
delivery_by_state.show(15)

In [0]:
# ANALYTICS: REVIEW SCORE AND PAYMENT DISTRIBUTION

review_dist = df.groupBy('review_score').agg(
  count('order_id').alias('count')
).orderBy('review_score')
print('=== REVIEW SCORE DISTRIBUTION ===')
review_dist.show()

pay_dist = payments.groupBy('payment_type').agg(
  count('order_id').alias('count')
).orderBy(desc('count'))
print('=== PAYMENT METHOD DISTRIBUTION ===')
pay_dist.show()


In [0]:
# Save master cleaned dataframe to your Volume
df.coalesce(1).write.mode('overwrite').option('header', True).csv(
    '/Volumes/workspace/default/dataset_olist/olist_cleaned'
)
print('Master df saved!')

# Verify
display(dbutils.fs.ls('/Volumes/workspace/default/dataset_olist/olist_cleaned/'))

In [0]:
df = spark.read.csv(
    '/Volumes/workspace/default/dataset_olist/olist_cleaned/',
    header=True, inferSchema=True
)

# Re-import needed functions
from pyspark.sql.functions import col, count, avg, desc
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import round as spark_round

print(f'Loaded: {df.count():,} rows')
df.printSchema()

### Machine Learning

In [0]:
# CREATE BINARY LABEL AND PREPARE FEATURES

from pyspark.sql.functions import col, when

# review_score 1,2,3 = Negative = label 0
# review_score 4,5   = Positive = label 1
ml_df = df.withColumn(
    'label',
    when(col('review_score') >= 4, 1).otherwise(0)
)

# Keep only numeric feature columns and label
feature_cols = ['delivery_days', 'total_price', 'freight_value',
                'payment_installments', 'item_count']

ml_df = ml_df.select(feature_cols + ['label']).dropna()

print(f'ML dataset size: {ml_df.count():,} rows')
print()
print('Label distribution (0=Negative review, 1=Positive review):')
ml_df.groupBy('label').count().orderBy('label').show()

In [0]:
# SPLIT DATA: 80% TRAIN (FOR GRID SEARCH CV) / 20% TEST

# CrossValidator performs its own internal k-fold validation on train_df,so a separate validation set is not needed here.
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print(f'Training samples (used by Grid Search CV) : {train_df.count():,}')
print(f'Test     samples (final evaluation only)  : {test_df.count():,}')
print()
print('IMPORTANT: test_df is locked until the very end.')
print('Grid Search finds the best params using cross-validation on train_df only.')

In [0]:
# BUILD PIPELINE, PARAM GRID, AND CROSS VALIDATOR

import os
# Set temp path for Spark ML on serverless cluster (must be set before CrossValidator)
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/default/dataset_olist'

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Step 1: Assemble feature columns into one vector
assembler = VectorAssembler(
    inputCols=['delivery_days', 'total_price', 'freight_value',
               'payment_installments', 'item_count'],
    outputCol='features'
)

# Step 2: Define the Random Forest classifier
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    seed=42
)

# Step 3: Chain into a pipeline
pipeline = Pipeline(stages=[assembler, rf])

# Step 4: Define the hyperparameter grid to search
# Grid Search will try ALL combinations:
# 3 numTrees x 3 maxDepth x 2 minInstancesPerNode = 18 combinations
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees,             [50, 100, 150])  \
    .addGrid(rf.maxDepth,             [4,  6,   8])    \
    .addGrid(rf.minInstancesPerNode,  [1,  5])         \
    .build()

# Step 5: Define the evaluator (optimise for F1 score)
evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='f1'
)

# Step 6: Create CrossValidator
# numFolds=3 means each of the 18 combinations is trained and
# evaluated 3 times on different folds -> 54 total model fits
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

print('Grid Search setup complete.')
print(f'Total parameter combinations : {len(param_grid)}')
print(f'Cross-validation folds       : 3')
print(f'Total model fits             : {len(param_grid) * 3}')
print()
print('Parameters being searched:')
print('  numTrees            : 50, 100, 150')
print('  maxDepth            : 4, 6, 8')
print('  minInstancesPerNode : 1, 5')

In [0]:
# RUN GRID SEARCH (fit CrossValidator on training data)

print('Running Grid Search with 3-fold Cross-Validation...')
print('This may take several minutes on Community Edition.')
print()

cv_model = cv.fit(train_df)   # Grid Search happens here

print('Grid Search complete!')
print()

# Extract the best pipeline model found by CV
best_model = cv_model.bestModel
best_rf    = best_model.stages[-1]   # Random Forest stage from pipeline

print('=== BEST HYPERPARAMETERS FOUND ===')
print(f'  numTrees            : {best_rf.getNumTrees}')
print(f'  maxDepth            : {best_rf.getMaxDepth()}')
print(f'  minInstancesPerNode : {best_rf.getMinInstancesPerNode()}')
print()

# Show CV scores for every combination tried
print('=== ALL COMBINATIONS TRIED (sorted by F1, best first) ===')
import pandas as pd

results = []
for params, score in zip(param_grid, cv_model.avgMetrics):
    results.append({
        'numTrees'           : params[rf.numTrees],
        'maxDepth'           : params[rf.maxDepth],
        'minInstances'       : params[rf.minInstancesPerNode],
        'avg_CV_F1'          : round(score, 4)
    })

results_df = pd.DataFrame(results).sort_values('avg_CV_F1', ascending=False)
print(results_df.to_string(index=False))

In [0]:
# FINAL EVALUATION ON TEST SET (best model only, run once)

print('=== FINAL TEST SET EVALUATION ===')
print(f'Best params: numTrees={best_rf.getNumTrees}, '
      f'maxDepth={best_rf.getMaxDepth()}, '
      f'minInstancesPerNode={best_rf.getMinInstancesPerNode()}')
print()

test_predictions = best_model.transform(test_df)

# Calculate all metrics
accuracy  = evaluator.setMetricName('accuracy').evaluate(test_predictions)
precision = evaluator.setMetricName('weightedPrecision').evaluate(test_predictions)
recall    = evaluator.setMetricName('weightedRecall').evaluate(test_predictions)
f1        = evaluator.setMetricName('f1').evaluate(test_predictions)

print('================================================')
print('        FINAL MODEL EVALUATION RESULTS         ')
print('================================================')
print(f'  numTrees            : {best_rf.getNumTrees}')
print(f'  maxDepth            : {best_rf.getMaxDepth()}')
print(f'  minInstancesPerNode : {best_rf.getMinInstancesPerNode()}')
print('  ----------------------------------------')
print(f'  Accuracy            : {accuracy:.4f}  ({accuracy*100:.1f}%)')
print(f'  Precision           : {precision:.4f}')
print(f'  Recall              : {recall:.4f}')
print(f'  F1 Score            : {f1:.4f}')
print('================================================')
print()
print('Sample predictions:')
test_predictions.select(
    'delivery_days', 'total_price', 'label', 'prediction'
).show(8)

In [0]:
# FEATURE IMPORTANCE

import pandas as pd

feature_names = ['delivery_days', 'total_price', 'freight_value',
                 'payment_installments', 'item_count']
importances   = best_rf.featureImportances.toArray()

imp_df = pd.DataFrame({
    'Feature'   : feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('=== FEATURE IMPORTANCE (Best Model) ===')
print(imp_df.to_string(index=False))
print()
print('KEY FINDING: delivery_days is the strongest predictor.')
print('Late delivery is the #1 cause of negative reviews on Olist.')

spark_imp = spark.createDataFrame(imp_df)
display(spark_imp)

Databricks visualization. Run in Databricks to view.

### Visualizations

In [0]:
# VISUALIZATION 1: BAR CHART - TOP 10 CATEGORIES BY REVENUE

revenue_by_cat = df.groupBy('product_category_name_english').sum('total_price')
top10 = revenue_by_cat.orderBy('sum(total_price)', ascending=False).limit(10)

display(top10)

Databricks visualization. Run in Databricks to view.

In [0]:
# VISUALIZATION 2: LINE CHART - MONTHLY ORDER VOLUME

from pyspark.sql.functions import date_trunc, col, to_timestamp

monthly = (df.withColumn('purchase_ts', to_timestamp(col('order_purchase_timestamp'), 'M/d/yyyy H:mm'))
             .groupBy(date_trunc('month', 'purchase_ts').alias('month'))
             .agg(count('*').alias('order_count'),
                  spark_sum('total_price').alias('monthly_revenue'))
             .orderBy(col('month')))

display(monthly)

Databricks visualization. Run in Databricks to view.

In [0]:
# VISUALIZATION 3: PIE CHART - PAYMENT METHOD DISTRIBUTION

pay_dist = df.groupBy('payment_installments').count().orderBy('count', ascending=False)

display(pay_dist)

Databricks visualization. Run in Databricks to view.

In [0]:
# DELIVERY TIME DISTRIBUTION

delivery_dist = df.groupBy('delivery_days').agg(
    count('order_id').alias('order_count')
).filter(col('delivery_days') <= 60).orderBy('delivery_days')
 
display(delivery_dist)

Databricks visualization. Run in Databricks to view.